In [43]:
import requests

def _prebuilt_placeholder(tool_data):
    def _run(*args, **kwargs):
        return {"message": "Prebuilt tool placeholder", "tool": tool_data.get("name")}
    return _run

def _custom_function_placeholder(tool_data):
    def _run(*args, **kwargs):
        return {"message": "Custom function placeholder", "tool": tool_data.get("name")}
    return _run

In [44]:
from langchain_core.tools import StructuredTool
from pydantic import create_model
import requests
from typing import Dict, Any, List

def _custom_api_placeholder(tool_def: Dict[str, Any]) -> List:
    """
    Convert custom_api tool definition to LangChain tool
    
    Returns:  List containing one LangChain StructuredTool
    """
    name = tool_def["name"]
    description = tool_def["description"]
    api_url = tool_def["api_url"]
    api_request_type = tool_def["api_request_type"]
    custom_message = tool_def.get("custom_message", "")
    input_schema = tool_def["input_schema"]
    
    # Build Pydantic model from input_schema
    fields = {}
    properties = input_schema.get("properties", {})
    
    for field_name, field_spec in properties.items():
        field_type_map = {
            "string": str,
            "number": float,
            "integer":  int,
            "boolean": bool
        }
        field_type = field_type_map.get(field_spec.get("type"), str)
        fields[field_name] = (field_type, None)
    
    # Fallback if no properties
    if not fields: 
        fields = {"_placeholder": (str, None)}
    
    InputModel = create_model(f"{name}_Input", **fields)
    
    # API call function
    def execute_api_call(**kwargs) -> Dict[str, Any]:
        try:
            # Remove placeholder if exists
            kwargs.pop("_placeholder", None)
            
            if api_request_type.upper() == "GET":
                resp = requests.get(api_url, params=kwargs, timeout=10)
            else:  # POST
                resp = requests.post(api_url, json=kwargs, timeout=10)
            
            return {
                "status_code": resp.status_code,
                "data": resp.json() if resp.content else {},
                "custom_message": custom_message
            }
        except Exception as e:
            return {
                "status_code": 500,
                "data": {"error":  str(e)},
                "custom_message": custom_message
            }
    
    # Create LangChain tool
    tool = StructuredTool.from_function(
        func=execute_api_call,
        name=name,
        description=description,
        args_schema=InputModel
    )
    
    return [tool]

In [45]:
from typing import List
from langchain_core.tools import StructuredTool
from manager import ToolRegistryManager

def build_langchain_tools(tool_ids: List[str]) -> List[StructuredTool]:
    """
    Given a list of tool IDs, return LangChain-compatible Tool objects.
    """
    manager = ToolRegistryManager()
    langchain_tools: List[StructuredTool] = []

    for tool_id in tool_ids:
        tool_data = manager.get_tool(tool_id)
        if not tool_data:
            print("tool is missing")
            continue

        tool_type = tool_data.get("type")
        name = tool_data.get("name")
        description = tool_data.get("description")

        # --- Create tools based on type ---
        if tool_type == "prebuilt":
            func = _prebuilt_placeholder(tool_data)
            tool = StructuredTool.from_function(
                func=func,
                name=name,
                description=description
            )
            langchain_tools.append(tool)

        elif tool_type == "custom_function":
            func = _custom_function_placeholder(tool_data)
            tool = StructuredTool.from_function(
                func=func,
                name=name,
                description=description
            )
            langchain_tools.append(tool)

        elif tool_type == "custom_api":
            # _custom_api_placeholder returns a list of tools
            tools = _custom_api_placeholder(tool_data)
            langchain_tools.extend(tools)

    return langchain_tools

In [48]:
tool_ids = ["agent_fe925649-e6ec-4002-8d7f-7374d07ffa7d","agent_54da582b-e8e6-4f98-ab19-bfaf0a5965b6"]

tools = build_langchain_tools(tool_ids)

In [59]:
response = tools[0].invoke({
  "order_id": "ORD123",
})
response

{'status_code': 200,
 'data': {'order_id': '{order_id}', 'item': 'the item is rphone'},
 'custom_message': 'Order information fetched successfully'}

In [58]:
response = tools[1].invoke({
    "order_id": "ORD123",
    "item": "phone"
})
response

{'status_code': 200,
 'data': {'received': 'ORD123', 'status': 'the order has been created'},
 'custom_message': 'Order created successfully'}

In [62]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from dotenv import load_dotenv
load_dotenv()


llm = ChatOpenAI(temperature=0)

agent = create_agent(model=llm, tools=tools)

response=agent.invoke({"messages": [("user", "Get info for order id ord123")]})
response

{'messages': [HumanMessage(content='Get info for order id ord123', additional_kwargs={}, response_metadata={}, id='152557b8-a82e-4443-9cb0-9bc48ef1c08b'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 95, 'total_tokens': 111, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-CwrKUkHXoQX1LckKXAWAbXgQd5ibq', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019bad99-362f-7a51-a5d1-49fe660281e2-0', tool_calls=[{'name': 'order_info', 'args': {'order_id': 'ord123'}, 'id': 'call_MxE5DA7RZTaSLEFT6kWpvJZ1', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 95, 'output_tokens': 1

In [ ]:
response['messages'][-1].content

In [64]:

response=agent.invoke({"messages": [("user", "create a new order, order idr123 and item is mouse")]})
response

{'messages': [HumanMessage(content='create a new order, order idr123 and item is mouse', additional_kwargs={}, response_metadata={}, id='2cafd77e-dcf2-46fb-8715-941e438929d9'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 101, 'total_tokens': 121, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-CwrLDvrX14nDfQ26R0QWmMT6DyDfD', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019bad99-e448-7e70-9069-b96ad7d5bf84-0', tool_calls=[{'name': 'create_order', 'args': {'order_id': 'r123', 'item': 'mouse'}, 'id': 'call_Xz2qBWVYtgWfsMRFQKzUk9uO', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata

In [ ]:
response['messages'][-1].content

'The order with ID r123 for the item "mouse" has been created successfully.'